# 🔀 Day 7 — Multi-Step Workflows
## Build the 4-Node Compliance Pipeline

---

## The idea

In Day 6 you answered one question with one LLM call.

Today an invoice arrives and four things need to happen in sequence:

```
Node 1: classify()     → What type of transaction is this?
Node 2: lookup_rate()  → What is the applicable GST rate?
Node 3: calculate()    → What is the tax amount?
Node 4: route()        → DB insert, or human review?
```

Each node does **one job** and passes its result forward in a shared `state` dict.
Only Node 1 calls the LLM. Nodes 2, 3, and 4 are pure Python.

---
## ⏱️ Time: 90 minutes

## Step 1 — Install and Import

In [ ]:
import json
import re
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()   # reads .env from this directory

client     = AzureOpenAI(
    azure_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT'),
    api_key        = os.getenv('AZURE_OPENAI_API_KEY'),
    api_version    = '2024-08-01-preview',
)
CHAT_MODEL = os.getenv('AZURE_OPENAI_CHAT_MODEL', 'gpt-4o')

print('✅ Ready')

## Step 2 — The State Dict

Every node reads from and writes to the same dict.
This is what connects the nodes — no global variables, no class.
Just a dict that grows as it passes through the pipeline.

In [ ]:
# This is what the state looks like at the START — only the inputs
state = {
    "invoice_text": "TechSolutions Pvt Ltd (Mumbai) invoiced DataCore Ltd (Bengaluru) "
                    "Rs 8,50,000 for software implementation services. Q3 FY2025.",
    "amount":       850000,
}

print("Initial state:")
print(json.dumps(state, indent=2, ensure_ascii=False))

## Step 3 — Node 1: classify()

The only node that calls the LLM.
It reads the invoice text and returns `transaction_type` and `confidence`.

We ask the model to return JSON so the output is structured and usable.

In [ ]:
def classify(state):
    """Node 1 — one LLM call. Classifies the invoice text."""

    prompt = (
        f"Classify this invoice and return ONLY valid JSON:\n\n{state['invoice_text']}\n\n"
        "JSON format: {\"transaction_type\": \""
        "B2B Intra-State|B2B Inter-State|Export of Services|Import RCM|Exempt|Works Contract|Unknown\", "
        "\"confidence\": 0.0-1.0, \"notes\": \"brief reason\"}"
    )

    raw = client.chat.completions.create(
        model      = CHAT_MODEL,
        messages   = [{"role": "user", "content": prompt}],
        temperature= 0.0,
        max_tokens = 150,
    ).choices[0].message.content

    # Parse the JSON response
    try:
        cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip(), flags=re.MULTILINE)
        result  = json.loads(cleaned)
        state.update({
            "transaction_type": result.get("transaction_type", "Unknown"),
            "confidence":       float(result.get("confidence", 0.5)),
            "classify_notes":   result.get("notes", ""),
        })
    except Exception:
        state.update({"transaction_type": "Unknown", "confidence": 0.3})

    return state


# Run Node 1
state = classify(state)

print(f"Transaction type : {state['transaction_type']}")
print(f"Confidence       : {state['confidence']:.2f}")
print(f"Notes            : {state['classify_notes']}")

## Step 4 — Node 2: lookup_rate()

A plain Python dict maps transaction types to GST rates.
Zero LLM calls. Instant. Deterministic.

**Activity 3:** Add a new transaction type to `RATES` below.

In [ ]:
# GST rates table — extend this in Activity 3
RATES = {
    "B2B Intra-State":    {"rate": 18.0, "split": "CGST 9% + SGST 9%", "section": "CGST Act §9"},
    "B2B Inter-State":    {"rate": 18.0, "split": "IGST 18%",           "section": "IGST Act §5"},
    "Export of Services": {"rate": 0.0,  "split": "Zero-rated (LUT)",   "section": "IGST Act §16(1)"},
    "Import RCM":         {"rate": 18.0, "split": "RCM",                "section": "CGST Act §9(3)"},
    "Exempt":             {"rate": 0.0,  "split": "Exempt",              "section": "CGST Schedule III"},
    "Works Contract":     {"rate": 18.0, "split": "IGST 18%",           "section": "Notification 11/2017"},
    # ← Activity 3: add your own transaction type here
}


def lookup_rate(state):
    """Node 2 — dict lookup. Zero LLM calls."""
    info = RATES.get(state["transaction_type"], {"rate": None, "split": "Unknown", "section": "—"})
    state.update({"rate": info["rate"], "rate_split": info["split"], "section": info["section"]})
    return state


# Run Node 2
state = lookup_rate(state)

print(f"GST rate : {state['rate']}%")
print(f"Split    : {state['rate_split']}")
print(f"Section  : {state['section']}")

## Step 5 — Node 3: calculate()

Pure Python arithmetic — the simplest node.
No LLM, no API call, no latency.

In [ ]:
def calculate(state):
    """Node 3 — pure Python math. Zero LLM calls."""
    if state.get("amount") and state.get("rate") is not None:
        tax   = round(state["amount"] * state["rate"] / 100, 2)
        total = round(state["amount"] + tax, 2)
        state.update({"tax_amount": tax, "total_amount": total})
    else:
        state.update({"tax_amount": None, "total_amount": None})
    return state


# Run Node 3
state = calculate(state)

print(f"Base amount : ₹{state['amount']:,.0f}")
print(f"GST @ {state['rate']}%   : ₹{state['tax_amount']:,.0f}")
print(f"Total       : ₹{state['total_amount']:,.0f}")

## Step 6 — Node 4: route()

Routing logic is Python — not a prompt.
An `if/else` is deterministic, testable, and auditable.

**Activity 2:** Change the threshold below from `0.78` to `0.60` and see what changes.

In [ ]:
THRESHOLD = 0.78   # ← Activity 2: change this


def route(state, threshold=THRESHOLD):
    """Node 4 — if/else routing. Zero LLM calls."""
    reasons = []
    if state.get("confidence", 0) < threshold:
        reasons.append(f"low confidence ({state['confidence']:.2f})")
    if state.get("transaction_type") == "Unknown":
        reasons.append("transaction type unresolved")
    if not state.get("amount"):
        reasons.append("invoice amount missing")
    if state.get("rate") is None:
        reasons.append("GST rate not found")

    state["route"]          = "human_review" if reasons else "db_insert"
    state["review_reasons"] = reasons
    return state


# Run Node 4
state = route(state)

icon = "✅" if state["route"] == "db_insert" else "⚠️"
print(f"{icon}  Route: {state['route'].upper()}")
if state["review_reasons"]:
    print(f"   Reasons: {', '.join(state['review_reasons'])}")

## Step 7 — Wire all 4 nodes into run_pipeline()

The pattern is the same for every node: `state = node(state)`.
Adding or removing a node is one line.

In [ ]:
def run_pipeline(invoice_text, amount, threshold=THRESHOLD):
    """Run all 4 nodes in sequence."""
    state = {"invoice_text": invoice_text, "amount": amount}

    for node in [classify, lookup_rate, calculate, route]:
        state = node(state)   # same pattern, every node

    return state


print("✅ run_pipeline() ready")

## Step 8 — Test with 3 Invoices

**Activity 1:** Add your own invoice to `TEST_INVOICES` before running this cell.

In [ ]:
TEST_INVOICES = [
    {
        "id":   "INV-001",
        "text": "TechSolutions Pvt Ltd (Mumbai) invoiced DataCore Ltd (Bengaluru) "
                "Rs 8,50,000 for software implementation services. Q3 FY2025.",
        "amount": 850000,
    },
    {
        "id":   "INV-002",
        "text": "GlobalServ India exports IT consulting to Acme Corp USA. "
                "Invoice USD 15,000. LUT filed FY2024-25.",
        "amount": 1250000,
    },
    {
        "id":   "INV-003",
        "text": "Hi, can you check the GST on recent payments? - Amit",
        "amount": 0,
    },
    # ← Activity 1: add your own invoice here
    # {
    #     "id":   "INV-004",
    #     "text": "Your invoice text here",
    #     "amount": 500000,
    # },
]


print("Running pipeline on all invoices...\n")
results = []

for inv in TEST_INVOICES:
    result = run_pipeline(inv["text"], inv["amount"])
    result["invoice_id"] = inv["id"]
    results.append(result)

    icon = "✅" if result["route"] == "db_insert" else "⚠️ "
    tax  = f"₹{result['tax_amount']:,.0f}" if result.get("tax_amount") else "—"
    print(f"{icon}  {inv['id']}  |  {result['transaction_type']:<22}  |  "
          f"{str(result['rate'])+'%':<8}  |  {tax:<14}  |  {result['route']}")

## Step 9 — The Full State Dict (Audit Trail)

In [ ]:
# Pick any result and print its full state
IDX = 0   # ← change this to 1 or 2 to see other results

r = results[IDX]
print(f"Full state dict for {r['invoice_id']}:")
print("-" * 50)
for k, v in r.items():
    if k != "invoice_text":
        print(f"  {k:<22} : {v}")

---
## Activity 4 — Add a 5th Node: summarise()

For invoices that route to `db_insert`, add a 5th node that asks GPT
to write a one-sentence compliance note.

**Template:**

```python
def summarise(state):
    """Node 5 — generate a compliance note for db_insert results."""
    if state["route"] != "db_insert":
        return state   # skip if not going to DB

    prompt = (
        f"Write a one-sentence compliance note for this transaction:\n"
        f"Type: {state['transaction_type']}  Rate: {state['rate']}%  "
        f"Section: {state['section']}  Tax: ₹{state['tax_amount']:,.0f}"
    )
    state["compliance_note"] = ___________________  # ← your code here
    return state
```

Then add `summarise` to the node list in `run_pipeline()`.

In [ ]:
# Activity 4: write your summarise() node here


---
## What you built

```
invoice_text + amount
      ↓
  classify()     → transaction_type, confidence     (1 LLM call)
      ↓
  lookup_rate()  → rate, section                    (0 LLM calls)
      ↓
  calculate()    → tax_amount, total_amount          (0 LLM calls)
      ↓
  route()        → db_insert OR human_review         (0 LLM calls)
      ↓
  state dict — the complete audit trail
```

**Day 8** adds memory — the pipeline will remember previous invoices
within the same session. If a follow-up says *'same vendor as last invoice'*,
the classifier can resolve it.

The pipeline doesn't change. Only the context available to Node 1 grows.